In [1]:
NAMA:Fahmiori Afnil
NIM:2411513033
TUGAS PEMPAR 3

SyntaxError: invalid syntax (1925668032.py, line 1)

In [ ]:
SOAL 1

In [ ]:
import time
import multiprocessing as mp
import matplotlib.pyplot as plt

def hitung_prima(n):
    # Fungsi CPU-bound untuk mencari bilangan prima hingga n
    count = 0
    for i in range(2, n):
        prime = True
        for j in range(2, int(i**0.5) + 1):
            if i % j == 0:
                prime = False
                break
        if prime:
            count += 1
    return count

def run_experiment_multiprocessing():
    n_value = 100000  # Sesuaikan ukuran beban kerja
    workers_list = [1, 2, 4, 8]
    waktu_list = []

    # Waktu sekuensial dasar
    start_seq = time.perf_counter()
    hitung_prima(n_value)
    baseline_time = time.perf_counter() - start_seq

    print("--- Memulai Eksperimen Speedup Skala ---")
    
    # Menggunakan metode default yang aman lintas platform (spawn/fork)
    for w in workers_list:
        start_time = time.perf_counter()
        with mp.Pool(processes=w) as pool:
            # Membagi beban kerja ke worker sejumlah w
            pool.map(hitung_prima, [n_value // w] * w)
        end_time = time.perf_counter()
        
        durasi = end_time - start_time
        waktu_list.append(durasi)
        print(f"Worker = {w} | Waktu Eksekusi: {durasi:.4f} detik")

    # Menghitung speedup (Waktu 1 worker / Waktu w worker)
    speedup_list = [waktu_list[0] / t for t in waktu_list]

    # Plot Speedup vs Jumlah Worker
    plt.figure(figsize=(8, 5))
    plt.plot(workers_list, speedup_list, marker='o', color='r', linestyle='-', label='Aktual')
    plt.plot(workers_list, workers_list, color='gray', linestyle='--', label='Teoretis (Linear)')
    plt.title('Analisis Speedup vs Jumlah Worker (Hukum Amdahl)')
    plt.xlabel('Jumlah Worker')
    plt.ylabel('Speedup')
    plt.legend()
    plt.grid(True)
    plt.show()

# Jalankan fungsi
if __name__ == '__main__':
    run_experiment_multiprocessing()

In [ ]:
Analisis Kurva Aktual vs Teoretis:
Mengapa kurva aktual tidak persis mengikuti kurva teoretis?

Overhead Pembuatan Proses (Process Overhead): Pembuatan proses baru (forking) dan komunikasi antar-proses membutuhkan sumber daya waktu dan CPU, terutama jika tugas komputasinya relatif cepat.

Keterbatasan Core Fisik: Jika jumlah worker melebihi jumlah core fisik CPU (misalnya menggunakan 8 worker pada mesin ber-core fisik 4), terjadi context switching yang memicu penurunan efisiensi.

Beban Kerja Lain: Proses latar belakang sistem operasi lain ikut memakai CPU, sehingga peningkatan jumlah worker tidak memberikan peningkatan speedup yang linier (sesuai hukum Amdahl yang dibatasi oleh bagian sekuensial dan overhead).

In [ ]:
SOAL 2

In [ ]:
import time
import random
import multiprocessing as mp

def worker_queue(queue, numbers):
    subtotal = sum(numbers)
    queue.put(subtotal)

def worker_value(shared_val, lock, numbers):
    subtotal = sum(numbers)
    with lock:
        shared_val.value += subtotal

def main_eksperimen_2():
    total_angka = 1_000_000
    num_processes = 4
    
    # Membagi data untuk 4 proses
    chunk_size = total_angka // num_processes
    data_chunks = [
        [random.randint(1, 100) for _ in range(chunk_size)] 
        for _ in range(num_processes)
    ]

    # --- (a) Versi Queue (Message Passing) ---
    start_q = time.perf_counter()
    queue = mp.Queue()
    processes_q = [
        mp.Process(target=worker_queue, args=(queue, data_chunks[i]))
        for i in range(num_processes)
    ]

    for p in processes_q: p.start()
    
    total_q = sum([queue.get() for _ in range(num_processes)])
    for p in processes_q: p.join()
    
    end_q = time.perf_counter()
    waktu_q = end_q - start_q

    # --- (b) Versi Value (Shared Memory + Lock) ---
    start_v = time.perf_counter()
    shared_val = mp.Value('i', 0)
    lock = mp.Lock()
    processes_v = [
        mp.Process(target=worker_value, args=(shared_val, lock, data_chunks[i]))
        for i in range(num_processes)
    ]

    for p in processes_v: p.start()
    for p in processes_v: p.join()
    
    total_v = shared_val.value
    end_v = time.perf_counter()
    waktu_v = end_v - start_v

    print(f"Hasil Queue : {total_q} | Waktu: {waktu_q:.4f} detik")
    print(f"Hasil Value : {total_v} | Waktu: {waktu_v:.4f} detik")

# Jalankan di Jupyter Notebook
main_eksperimen_2()

In [ ]:
SOAL 3

In [ ]:
import time
import multiprocessing as mp

def cpu_bound_task(n):
    # Simulasi tugas CPU-bound (misal: faktorial/hitung berat)
    res = 1
    for i in range(1, n):
        res += i
    return res

def producer(queue, num_items):
    for i in range(num_items):
        data = 5_000_000 + i
        queue.put(data)
    # Masukkan sinyal selesai (Sentinel)
    for _ in range(2): # Sebanyak jumlah consumer
        queue.put(None)

def consumer(queue, name):
    while True:
        item = queue.get()
        if item is None:
            break
        cpu_bound_task(item)

def main_eksperimen_3():
    queue = mp.Queue()
    num_items = 10

    p_prod = mp.Process(target=producer, args=(queue, num_items))
    p_cons1 = mp.Process(target=consumer, args=(queue, "Consumer-1"))
    p_cons2 = mp.Process(target=consumer, args=(queue, "Consumer-2"))

    start_time = time.perf_counter()

    p_prod.start()
    p_cons1.start()
    p_cons2.start()

    p_prod.join()
    p_cons1.join()
    p_cons2.join()

    end_time = time.perf_counter()
    print(f"Waktu Eksekusi Multiprocessing CPU-bound: {end_time - start_time:.4f} detik")

# Jalankan di Jupyter Notebook
main_eksperimen_3()

In [ ]:
SOAL 4

In [ ]:
Saya akan memilih Manager() ketika mengelola struktur data kompleks yang heterogen seperti dict atau list antar-proses secara bersamaan, karena menyediakan kemudahan penggunaan (ease of use) yang tinggi tanpa perlu skema serialization manual yang rumit. Namun, trade-off-nya adalah penurunan performa akibat overhead komunikasi server-process yang dikelola oleh Manager.

Sebaliknya, saya akan memilih Value atau Array ketika fokus utama adalah kecepatan maksimal dan penggunaan memori yang efisien untuk tipe data primitif sederhana. Walaupun implementasinya sedikit lebih kaku dan membutuhkan pengelolaan Lock secara manual untuk menghindari race condition, performa eksekusinya jauh lebih cepat karena mengakses memori bersama secara langsung (shared memory). 
Oleh karena itu, pemilihan bergantung pada kebutuhan prioritas antara kemudahan penulisan kode atau performa waktu eksekusi yang optimal.